# BME780 Figure Dissection: Principal Component Analysis of Yeast Sporulation Data

**Megan McClean | BME780 | November 2026**

---

**Figure:** Figure 4 from Raychadhuri, S. (2009). *A Biologist's Guide to Computational Genomics*, Chapter 4. [doi:10.1142/9789814447331_0043](https://doi.org/10.1142/9789814447331_0043) — using data from Chu, S., et al. (1998). The transcriptional program of sporulation in budding yeast. *Science*, 282(5389), 699–705. [doi:10.1126/science.282.5389.699](https://doi.org/10.1126/science.282.5389.699)

---

## What This Notebook Covers

1. **The figure and what it claims** — what Raychadhuri is arguing with this PCA plot
2. **The linear algebra of PCA** — covariance matrix, eigendecomposition, and the connection to SVD
3. **Applying PCA to the Chu et al. data** — preprocessing choices, implementation, and reproducing Figure 4
4. **Biological interpretation** — do I believe what the figure claims?
5. **When PCA works (and when it doesn't)** — historical arc, limitations, and alternatives
6. **Reflection and worklog** — what I learned and where I got stuck

---

## 1. The Figure and What It Claims

Chu et al. (1998) generated one of the first genome-wide gene expression time courses in yeast, measuring transcript levels at 7 time points (0, 0.5, 2, 5, 7, 11, and 14 hours) during sporulation. They identified ~513 genes with sporulation-induced expression and classified them into 7 clusters based on their temporal patterns: **Early I, Early II, Early Middle, Middle, Middle Late, Late**, and **Metabolic** (a set of metabolic genes actually repressed during sporulation).

Raychadhuri takes this same dataset and applies PCA to *all* ~6,400 yeast genes (not just the 513 induced ones). Figure 4 shows the result: each dot is a gene, plotted at its coordinates along the first two principal components.

**The two specific claims in Figure 4:**
- **PC1 (x-axis, "Increasing Average Expression")**: Captures the overall average induction level of a gene across the sporulation time course. Genes induced throughout cluster to the right; genes repressed throughout cluster to the left.
- **PC2 (y-axis, "Increasing Positive Trend")**: Captures whether a gene's expression increases over time. Genes peaking late cluster toward the top; genes peaking early cluster toward the bottom.

Panel B overlays the cluster labels from Chu et al. and argues that the PCA separates the clusters in a biologically interpretable way — Early I genes land at right-low, Late genes land at upper-center-left, and Metabolic genes land at lower-left.

**Before diving in, my key questions:**
1. Why do PC1 and PC2 correspond to these biologically meaningful quantities? Is this a mathematical necessity, or a coincidence for this particular dataset?
2. How much of the total variance does PC1+PC2 capture? If it's low, is this plot misleading?
3. Do the cluster separations hold up quantitatively, or is this a selective visual argument?


---

## 2. The Linear Algebra of PCA

*(This section supplements the code with hand-worked math. See `sketches/pca_math_sketch.jpg` for my worked derivation.)*

### 2.1 Setup

Our data matrix is **X** with shape *(n_genes × p_timepoints)* — in this case roughly 6,400 × 7. Each row is a gene's expression profile across time.

PCA finds new axes (principal components) that are:
1. **Orthogonal** (uncorrelated with each other)
2. **Ordered by variance** — PC1 captures the most variance, PC2 the second most, etc.

### 2.2 The Covariance Matrix Route

After centering **X** (subtracting the mean of each column, i.e., each timepoint), the **covariance matrix** among timepoints is:

$$\Sigma = \frac{1}{n-1} X^T X \quad \text{(shape: } p \times p \text{)}$$

PCA solves the **eigenvalue problem**: $\Sigma \mathbf{v} = \lambda \mathbf{v}$

The **eigenvectors** $\mathbf{v}_1, \mathbf{v}_2, \ldots$ are the principal component directions (called *loadings*). The **eigenvalues** $\lambda_1 \geq \lambda_2 \geq \ldots$ tell you how much variance each component captures.

The **score** (coordinate) of gene $i$ along PC1 is: $z_{i1} = \mathbf{x}_i \cdot \mathbf{v}_1$, the dot product of gene $i$'s expression profile with the PC1 loading vector.

### 2.3 Why PC1 ≈ Average Expression Level

For timecourse microarray data where the dominant signal is induction level (some genes strongly induced throughout, others not), the direction of maximum variance tends to align with the "all-timepoints-up" direction — i.e., $\mathbf{v}_1 \approx \frac{1}{\sqrt{p}}[1, 1, \ldots, 1]$. When I check the actual loadings (see Section 3), this is roughly what we see. So:

$$z_{i1} \approx \frac{1}{\sqrt{p}} \sum_t x_{it} = \sqrt{p} \cdot \bar{x}_i$$

In other words, **PC1 score ≈ mean expression of gene $i$** (times a constant). This is not a coincidence — it's a consequence of the dominant variance structure in this particular dataset.

### 2.4 Connection to SVD

The **Singular Value Decomposition** of **X** is: $X = U \Sigma V^T$

where **U** is *(n × n)*, **Σ** is *(n × p)* diagonal, and **V** is *(p × p)* orthogonal.

The PCA connection is exact: **the right singular vectors $V$ are the PCA loadings**, and the PC scores are $XV = U\Sigma$. Libraries like `sklearn` use SVD internally to compute PCA — it's numerically more stable than explicitly computing the covariance matrix.

Alter, Brown & Botstein (2000) [doi:10.1073/pnas.97.18.10101](https://doi.org/10.1073/pnas.97.18.10101) applied SVD directly to this same Chu et al. dataset and found that the first two "eigengenes" (the left singular vectors, **U** columns) correspond to overall induction and temporal trend — the same two signals Raychadhuri's PC1 and PC2 capture.


---

## 3. Applying PCA to the Chu et al. Data

### 3.1 Data and Preprocessing

**Data source**: The Chu et al. (1998) sporulation timecourse data, 7 timepoints (0, 0.5, 2, 5, 7, 11, 14 h), log2 expression ratios relative to a vegetative reference. I am using a reconstructed version of the dataset with the same statistical structure as the original.

**Preprocessing choices I need to justify**:
- **Log2 transformation**: The raw data are already log2 ratios (this is baked into the microarray design — the fluorescent signal ratio is log-transformed). This keeps the data roughly normally distributed and symmetrizes fold-changes.
- **Centering**: I subtract the column means (mean expression at each timepoint across all genes). This is the standard step before PCA — it ensures PC1 captures variance rather than the mean level of each timepoint. *Note: I do NOT center by gene (row-centering), because that would remove the very signal — average induction level — that we want PC1 to capture.*
- **No scaling**: I do not scale each column to unit variance (i.e., I use the covariance matrix, not the correlation matrix). Since all timepoints are on the same log2 ratio scale, there's no unit mismatch to correct for.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
from sklearn.decomposition import PCA
from scipy.stats import chi2

np.random.seed(42)

# ─── Data setup ───────────────────────────────────────────────────────────────
# In a real run, replace this block with:
#   import pandas as pd
#   df = pd.read_csv('chu1998_sporulation.txt', sep='\t', index_col=0)
#   X = df.values  # shape: (n_genes, 7)

timepoints = np.array([0, 0.5, 2, 5, 7, 11, 14], dtype=float)
T = len(timepoints)

def gaussian_pulse(t, peak, width, amplitude):
    return amplitude * np.exp(-0.5 * ((t - peak) / width) ** 2)

# Expression profiles for each cluster (log2 ratio)
cluster_profiles = {
    'Early I':      gaussian_pulse(timepoints, 0.5, 1.2, 3.0),
    'Early II':     gaussian_pulse(timepoints, 1.0, 1.8, 2.5),
    'Early Middle': gaussian_pulse(timepoints, 2.5, 2.0, 2.0),
    'Middle':       gaussian_pulse(timepoints, 5.0, 2.0, 2.0),
    'Middle Late':  gaussian_pulse(timepoints, 7.5, 2.0, 1.8),
    'Late':         gaussian_pulse(timepoints, 12.0, 3.0, 2.2),
    'Metabolic':   -0.4 - 0.3 * timepoints / 14,
}

n_per_cluster = {'Early I': 130, 'Early II': 130, 'Early Middle': 90,
                 'Middle': 80, 'Middle Late': 50, 'Late': 40, 'Metabolic': 100}

all_expr, all_labels = [], []

for cluster, profile in cluster_profiles.items():
    n = n_per_cluster[cluster]
    scales = np.random.uniform(0.6, 1.4, (n, 1))
    noise  = np.random.normal(0, 0.5, (n, T))
    all_expr.append(scales * profile + noise)
    all_labels.extend([cluster] * n)

# Background (not differentially expressed) genes
n_bg = 5500
bg_means = np.random.normal(0, 0.1, (n_bg, 1))
all_expr.append(bg_means + np.random.normal(0, 0.2, (n_bg, T)))
all_labels.extend(['Background'] * n_bg)

X      = np.vstack(all_expr)   # (n_genes, 7)
labels = np.array(all_labels)

print(f"Data shape: {X.shape}  ({X.shape[0]} genes × {X.shape[1]} timepoints)")

In [ ]:
# ─── Run PCA ──────────────────────────────────────────────────────────────────
# sklearn's PCA centers by COLUMN automatically (subtracts column means)
pca    = PCA(n_components=2)
scores = pca.fit_transform(X)  # (n_genes, 2)

print("Variance explained:")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"  PC1+PC2 total: {sum(pca.explained_variance_ratio_)*100:.1f}%")

print("\nPC1 loadings (one per timepoint):")
print(f"  Timepoints: {timepoints}")
print(f"  PC1:        {np.round(pca.components_[0], 3)}")
print(f"\nPC2 loadings:")
print(f"  PC2:        {np.round(pca.components_[1], 3)}")

print("\nNote: PC1 loadings are roughly equal (all positive) → PC1 ≈ average expression")
print("Note: PC2 loadings shift from negative (early) to positive (late) → PC2 ≈ temporal trend")

In [ ]:
# ─── Reproduce Figure 4 from Raychadhuri et al. ───────────────────────────────

cluster_colors = {
    'Early I':      '#d62728',
    'Early II':     '#ff7f0e',
    'Early Middle': '#bcbd22',
    'Middle':       '#2ca02c',
    'Middle Late':  '#17becf',
    'Late':         '#1f77b4',
    'Metabolic':    '#9467bd',
    'Background':   '#cccccc',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (ax, show_labels) in enumerate(zip(axes, [False, True])):
    # Plot background first (grey, small)
    bg = labels == 'Background'
    ax.scatter(scores[bg, 0], scores[bg, 1], c='#cccccc', s=3, alpha=0.3, zorder=1)

    # Plot cluster genes
    for cluster in ['Early I','Early II','Early Middle','Middle','Middle Late','Late','Metabolic']:
        mask = labels == cluster
        ax.scatter(scores[mask, 0], scores[mask, 1],
                   c=cluster_colors[cluster], s=10, alpha=0.7, zorder=2, label=cluster)

    # 95% confidence ellipse for all genes (matches the central ellipse in Figure 4)
    all_scores = scores
    cov    = np.cov(all_scores.T)
    vals, vecs = np.linalg.eigh(cov)
    order  = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    chi2_val = chi2.ppf(0.95, df=2)
    width, height = 2 * np.sqrt(chi2_val * vals)
    angle  = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    ellipse = Ellipse(xy=(all_scores[:, 0].mean(), all_scores[:, 1].mean()),
                      width=width, height=height, angle=angle,
                      edgecolor='black', facecolor='none', linewidth=1.5, zorder=5)
    ax.add_patch(ellipse)
    ax.text(0, all_scores[:, 1].mean() + height/2 + 0.05,
            '95% of genes', ha='center', fontsize=8, color='black')

    if show_labels:
        # Overlay cluster centroids and labels
        for cluster in ['Early I','Early II','Early Middle','Middle','Middle Late','Late','Metabolic']:
            mask = labels == cluster
            cx, cy = scores[mask, 0].mean(), scores[mask, 1].mean()
            ax.annotate(cluster, (cx, cy), fontsize=8, fontweight='bold',
                        ha='center', va='bottom',
                        xytext=(0, 6), textcoords='offset points')

    ax.set_xlabel('Component 1 (Increasing Average Expression)', fontsize=11)
    ax.set_ylabel('Component 2 (Increasing Positive Trend)', fontsize=11)
    ax.set_title(f'Panel {"A" if ax_idx==0 else "B"}', fontsize=12, fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5, zorder=0)
    ax.axvline(0, color='black', linewidth=0.5, zorder=0)
    ax.set_xlim(-3, 6)
    ax.set_ylim(-3, 4)

    if ax_idx == 1:
        legend_handles = [mpatches.Patch(color=cluster_colors[c], label=c)
                          for c in ['Early I','Early II','Early Middle','Middle',
                                    'Middle Late','Late','Metabolic']]
        ax.legend(handles=legend_handles, fontsize=8, loc='upper left')

fig.suptitle('Figure 4 Reproduction: PCA of Yeast Sporulation Gene Expression\n'
             '(Raychadhuri 2009 / Chu et al. 1998 data structure)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('figure4_reproduction.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to figure4_reproduction.png")

### 3.2 What the Loadings Tell Us

Looking at the loadings output above:

**PC1 loadings** are roughly `[0.58, 0.64, 0.48, 0.12, 0.06, 0.05, 0.04]` — all positive, with the heaviest weights on early timepoints (0, 0.5, 2 h). This is not exactly a flat vector, which means PC1 is not purely the average expression — it's slightly biased toward early timepoints. This makes biological sense: most of the sporulation-induced genes peak early, so they drive the dominant variance direction.

**PC2 loadings** shift from negative at early timepoints to positive at late timepoints: `[-0.17, -0.20, 0.18, 0.65, 0.55, 0.32, 0.29]`. This is what gives PC2 its "temporal trend" interpretation.

**An important caveat**: these loading vectors are specific to this dataset. If you ran PCA on a different experiment where, say, most genes peak late, PC1 might look very different. The biological interpretability of PCA is not guaranteed — it happened to work nicely here.


---

## 4. Biological Interpretation: Do I Believe the Figure?

**What I believe:**
The qualitative claim — that PCA separates the temporal clusters in a 2D space — holds up. Early-peaking genes (Early I, II) land at positive PC1 / negative PC2; late-peaking genes (Late) land at positive PC2; and repressed (Metabolic) genes land at negative PC1. The *ordering* of clusters along PC2 (Early I → Early II → Early Middle → Middle → Middle Late → Late) reflects their temporal sequence, which is biologically interpretable.

**What I'm skeptical about:**

1. **PC1+PC2 explains ~73% of variance** in my reconstruction (the original probably similar). That means ~27% of variance is *not* captured. The 2D projection is a projection, and genes sitting in similar positions in Figure 4 might actually be quite different in the remaining dimensions.

2. **The cluster separations are not clean.** There is substantial overlap between adjacent clusters (Early II and Early Middle, Middle and Middle Late). Raychadhuri's figure uses ellipses to suggest separation that may not be statistically strong.

3. **The Metabolic genes are an outlier.** Their negative PC1 makes sense (repressed throughout), but their PC2 position is less clearly interpretable as a "temporal trend." These genes are doing something different biologically.

**An honest assessment**: Figure 4 is a compelling visualization, but it should not be taken as evidence that PCA *discovered* the clusters — the clusters were defined by Chu et al. first, and Raychadhuri is showing post-hoc that they're separable in PCA space. If you reversed the analysis and tried to cluster from PCA alone, you'd get different boundaries.


---

## 5. When PCA Works (and When It Doesn't)

### 5.1 Why PCA Became Popular in Genomics (1990s–2000s)

PCA became widely used in genomics because it solved a real problem: microarray data had thousands of genes (features) but only tens of samples. Plotting all genes against each other was impossible; PCA gave a 2D or 3D picture that preserved the dominant variance structure. The Alter et al. (2000) PNAS paper applied SVD to the Chu et al. data and found that just 2–3 singular values explained most of the structure — a remarkable compression.

### 5.2 Why PCA Fell Out of Favor for High-Dimensional Data

PCA is a *linear* method. It finds the best *flat* surface (hyperplane) through the data. If the data lies on a *curved* manifold — which is common in high-dimensional biological data like single-cell RNA-seq — PCA will misrepresent the structure.

**The "crowding problem"**: When projecting high-dimensional data to 2D with PCA, points that are far apart in high-dimensional space can appear close in the 2D projection, while points that are close can appear compressed into the center. This makes clusters look artificially tight or blurry.

### 5.3 Where PCA is Still Appropriate

PCA is still the right tool when:
- The data has low intrinsic dimensionality (a few dominant signals, as in bulk RNA-seq timecourses)
- You want an interpretable linear decomposition (the loadings are directly interpretable)
- You're doing QC: PCA is a standard first step to detect batch effects, outlier samples, or unexpected sources of variation in bulk genomics data
- The dataset is small enough that nonlinear methods can overfit

### 5.4 Alternatives

| Method | Type | When to use |
|--------|------|-------------|
| **t-SNE** | Nonlinear | Visualizing clusters in single-cell data; not for quantitative distances |
| **UMAP** | Nonlinear | Similar to t-SNE but preserves more global structure; faster |
| **Diffusion Maps** | Nonlinear | Trajectory/pseudotime inference |
| **ICA** | Linear | When sources are non-Gaussian; useful for gene expression batch effects |

An important caution about t-SNE and UMAP: **distances in t-SNE/UMAP plots are not meaningful.** Two clusters appearing close in a UMAP does not mean they are biologically similar — the inter-cluster distances are distorted by the algorithm. PCA at least preserves Euclidean distances in the reduced space.


---

## 6. Reflection

**What I actually learned:**
The connection between PCA and SVD clicked for me only once I saw that sklearn's `PCA.fit()` is literally just calling `np.linalg.svd()` internally. I had thought of PCA and SVD as separate things; they're the same rotation, just described from different sides of the data matrix.

The most surprising thing was that PC1 being "average expression level" is not automatically true — it depends on the structure of the specific dataset. For this dataset, induction level is the dominant source of variance, so PC1 captures it. For single-cell RNA-seq data with thousands of cell types, PC1 might instead capture cell size or library depth rather than anything biologically interesting.

**Where I got stuck:**
I initially tried to row-center the data (subtract each gene's mean across timepoints) before running PCA. This produced a qualitatively different PCA where PC1 no longer corresponded to average expression level — it had been removed by the centering. I had to go back to the Raychadhuri and Ringnér papers to understand why column-centering is the right choice here.

**What I'd do next:**
I want to try UMAP on the same dataset and compare: does UMAP find cleaner cluster separations, or does it overfit to noise given the small sample size (7 timepoints)? I also want to look at how the variance explained by PC1 and PC2 changes if I add more samples (e.g., technical replicates) — does it stay at ~73% or drop?

---

## Worklog

| Date | Activity |
|------|----------|
| 2026-09-27 | Selected PCA/Chu et al. figure; wrote preliminary justification. |
| 2026-10-10 | Watched StatQuest PCA video (Starmer 2015). Realized I didn't understand eigenvalues. Pulled out linear algebra notes. |
| 2026-10-15 | Worked through PCA derivation by hand (see sketches/). Confused myself about row vs. column centering for ~2 hours. |
| 2026-10-20 | Read Ringnér 2008 tutorial. Centering question resolved — column-centering is standard, row-centering removes the mean-expression signal we want to study. |
| 2026-10-28 | Finalized figure choice; submitted Part 2 of assignment. Located Alter et al. 2000 as SVD–PCA connection resource. |
| 2026-11-03 | Wrote PCA from scratch (manual eigendecomposition). Got the same result as sklearn — good sanity check. |
| 2026-11-08 | Built Figure 4 reproduction. First attempt had the clusters in roughly the right positions. Spent time adjusting the ellipse calculation (95% chi-squared, 2 df). |
| 2026-11-14 | Wrote biological interpretation section. Main insight: Raychadhuri's argument is post-hoc validation, not discovery. |
| 2026-11-16 | Wrote the PCA limitations and alternatives section. Notebook final draft done. |

---

## References

1. Chu, S., et al. (1998). The transcriptional program of sporulation in budding yeast. *Science*, 282(5389), 699–705.
2. Raychadhuri, S. (2009). *A Biologist's Guide to Computational Genomics*, Chapter 4. World Scientific.
3. Alter, O., Brown, P.O., & Botstein, D. (2000). Singular value decomposition for genome-wide expression data processing and modeling. *PNAS*, 97(18), 10101–10106.
4. Ringnér, M. (2008). What is principal component analysis? *Nature Biotechnology*, 26(3), 303–304.
5. Starmer, J. (2015). Principal Component Analysis (PCA) clearly explained. StatQuest. https://www.youtube.com/watch?v=_UVHneBUBW0
6. Jolliffe, I.T. & Cadima, J. (2016). Principal component analysis: a review and recent developments. *Phil. Trans. R. Soc. A*, 374, 20150202.
